# 03: Privacy options on BloodMNIST (alpha 0.5)

Runs FedAvg with client-level DP (3 noise levels), sample-level DP-SGD (Opacus), the simulated secure aggregation, and a membership-inference check on every run. Same settings as notebook 01 (10 sites, 30 rounds, 2 local epochs, lr 0.02, momentum 0.9). Turn on a T4 GPU first. About 25 minutes, most of it the DP-SGD run.

Guarantees: client-level DP hides whether a whole site took part. Sample-level DP-SGD hides one image inside a site. Secure aggregation here is SIMULATED masking, not cryptography.

In [ ]:
!nvidia-smi -L
!rm -rf /content/fedsurveil && git clone -q https://github.com/Nathaniel47/fedsurveil /content/fedsurveil
%cd /content/fedsurveil
!pip install -q -r requirements-ml.txt 2>&1 | tail -1
!python -m fedsurveil.data.fetch
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.__version__, torch.cuda.is_available())

In [ ]:
import glob, json, os, statistics, subprocess, sys
BASE = ["--dataset", "bloodmnist", "--rounds", "30", "--lr", "0.02", "--momentum", "0.9", "--local-epochs", "2", "--alpha", "0.5", "--mia"]

def go(*args):
    subprocess.run([sys.executable, "-m", "fedsurveil.fl.simulate", *BASE, *args], check=True, capture_output=True)

# Pilot: clip so loosely that nothing is clipped, only to measure typical update norms.
go("--seed", "0", "--rounds", "5", "--dp-clip", "1000000")
pilot = [json.loads(l) for l in open(glob.glob("runs/*seed0.jsonl")[0])]
norms = [r["mean_update_norm"] for r in pilot if r["type"] == "round"]
CLIP = round(statistics.median(norms), 2)
print("update norms", [round(n, 2) for n in norms], "-> clip C =", CLIP)
for f in glob.glob("runs/*.jsonl"):
    os.remove(f)

In [ ]:
for seed in [0, 1, 2]:
    go("--seed", str(seed))  # plain FedAvg, only the membership check
    go("--seed", str(seed), "--secagg")  # simulated secure aggregation
    for sigma in [0.3, 1.0, 3.0]:
        go("--seed", str(seed), "--dp-clip", str(CLIP), "--dp-noise", str(sigma))
    print("seed", seed, "done")

In [ ]:
go("--seed", "0", "--site-dp-noise", "1.0", "--site-dp-clip", "1.0")  # slow

In [ ]:
# Compact copy of every run: final-round numbers and the privacy fields.
for p in sorted(glob.glob("runs/*.jsonl")):
    rows = [json.loads(line) for line in open(p)]
    rounds = [r for r in rows if r["type"] == "round"]
    summary = [r for r in rows if r["type"] == "summary"]
    last = rounds[-1]
    print(
        "RUN", os.path.basename(p),
        json.dumps({"acc": round(last["global_acc"], 4), "f1": round(last["macro_f1"], 4),
                    "worst_site": round(last["site_acc_spread"]["min"], 4),
                    "eps_client": last.get("eps_client"), "eps_sample": last.get("eps_sample_max"),
                    "clipped": last.get("clipped_fraction"), "bytes_up": last["bytes_up"],
                    "mia": summary[0]["mia_auroc"] if summary else None}),
    )